# 🏦 Credit Risk Analysis


## 📂 Dataset
**Source**: [Give Me Some Credit – Kaggle](https://www.kaggle.com/c/GiveMeSomeCredit/data)
**File**: `cs-training.csv`



##Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

## 📥 Load Dataset

In [2]:
df = pd.read_csv('cs-training.csv', index_col=0)
df.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


## 🧹 Preprocess Data

In [3]:
# Fill missing values with median
df.fillna(df.median(), inplace=True)
df.isnull().sum()

,0
SeriousDlqin2yrs,0
RevolvingUtilizationOfUnsecuredLines,0
age,0
NumberOfTime30-59DaysPastDueNotWorse,0
DebtRatio,0
MonthlyIncome,0
NumberOfOpenCreditLinesAndLoans,0
NumberOfTimes90DaysLate,0
NumberRealEstateLoansOrLines,0
NumberOfTime60-89DaysPastDueNotWorse,0


## ⚙️ Feature Engineering

In [4]:
# Rename columns for readability
df.columns = ['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse',
              'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans',
              'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines',
              'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']

# Create additional features
df['IncomePerDebt'] = df['MonthlyIncome'] / (df['DebtRatio'] + 1)
df['TotalPastDue'] = df['NumberOfTime30-59DaysPastDueNotWorse'] + df['NumberOfTimes90DaysLate'] + df['NumberOfTime60-89DaysPastDueNotWorse']
df.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,IncomePerDebt,TotalPastDue
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0,5058.286410,2
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0,2317.546266,0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0,2803.393701,2
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0,3185.175438,0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0,62041.570731,1


## 🧪 Train-Test Split & SMOTE

In [5]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_scaled, y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

## 🤖 Model Training

In [6]:
# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# Gradient Boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)

# XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)

## 📊 Model Evaluation

In [7]:
def evaluate_model(y_true, y_pred, name):
    print(f"\n{name} Model")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred))
    print(f"ROC AUC Score: {roc_auc_score(y_true, y_pred):.4f}")

evaluate_model(y_test, rf_preds, "Random Forest")
evaluate_model(y_test, gb_preds, "Gradient Boosting")
evaluate_model(y_test, xgb_preds, "XGBoost")


Random Forest Model
[[26665  1260]
 [ 2018 26047]]
              precision    recall  f1-score   support

           0       0.93      0.95      0.94     27925
           1       0.95      0.93      0.94     28065

    accuracy                           0.94     55990
   macro avg       0.94      0.94      0.94     55990
weighted avg       0.94      0.94      0.94     55990

ROC AUC Score: 0.9415

Gradient Boosting Model
[[24666  3259]
 [ 4161 23904]]
              precision    recall  f1-score   support

           0       0.86      0.88      0.87     27925
           1       0.88      0.85      0.87     28065

    accuracy                           0.87     55990
   macro avg       0.87      0.87      0.87     55990
weighted avg       0.87      0.87      0.87     55990

ROC AUC Score: 0.8675

XGBoost Model
[[26943   982]
 [ 2692 25373]]
              precision    recall  f1-score   support

           0       0.91      0.96      0.94     27925
           1       0.96      0.90      

## ✅ Outcome
This predictive system flags **high-risk customers** and helps financial institutions:
- Make informed lending decisions
- Reduce **loan default rates**
- Improve credit risk strategy